# Week 8 Exercise

## Agentic Workflow with Code Generation Agents

This notebook shows an example of two Agents (Senior Architect and Senior Developer) working
in sync to ship a feature in zip format containing the code to fulfill the user request.


In [ ]:
# Imports, environment and constants
import os
import json
import zipfile
import shutil
import gradio as gr
from openai import OpenAI
from dotenv import load_dotenv

load_dotenv(override=True)
api_key = os.getenv('OPENAI_API_KEY')

MODEL = "gpt-5.4-mini"

SENIOR_ARCHITECT_AGENT_SYSTEM_PROMPT = """
You are an expert Ruby on Rails software architect.
Given a user's feature request, determine the necessary files to build it.
Output a JSON object containing a 'files' array. Each item in the array must have a 'filename' (string) and a 'description' (string detailing the specific code logic needed)."""

SENIOR_DEVELOPER_AGENT_SYSTEM_PROMPT = """
You are Ruby on Rails senior developer.
Write the code for the requested file based on the description.
Return ONLY the raw code. Do not wrap the code in markdown blocks (e.g., ```python) and do not include any conversational text."""



In [ ]:
# Architect Agent generates the Plan (using JSON mode)
def architect_agent(query, logs):
    client = OpenAI(api_key=api_key)
    plan_response = client.chat.completions.create(
        model=MODEL,
        response_format={ "type": "json_object" },
        messages=[
            {
                "role": "system", 
                "content": SENIOR_ARCHITECT_AGENT_SYSTEM_PROMPT
            },
            {"role": "user", "content": query}
        ]
    )
    
    plan_content = plan_response.choices[0].message.content
    plan_json = json.loads(plan_content)
    files_to_create = plan_json.get("files", [])
    
    logs += f"Plan created! {len(files_to_create)} files to generate.\n\n"
    for f in files_to_create:
        logs += f" - {f['filename']}: {f['description'][:50]}...\n"

    return files_to_create, logs


In [ ]:
# Senior Developer Agent executes the Plan (Generate Code Files)
def seniordev_agent(filename, description, output_dir):
  client = OpenAI(api_key=api_key)

  code_response = client.chat.completions.create(
      model=MODEL,
      messages=[
          {
              "role": "system", 
              "content": SENIOR_DEVELOPER_AGENT_SYSTEM_PROMPT
          },
          {
              "role": "user", 
              "content": f"Filename: {filename}\nDescription: {description}"
          }
      ]
  )
  
  raw_code = code_response.choices[0].message.content
  
  # Failsafe: Clean up markdown codeblocks if the LLM still outputs them
  if raw_code.startswith("```"):
      lines = raw_code.split("\n")
      raw_code = "\n".join(lines[1:-1])
  
  # Save the file
  file_path = os.path.join(output_dir, filename)
  
  # Ensure subdirectories exist if the LLM suggests nested files
  os.makedirs(os.path.dirname(file_path) or ".", exist_ok=True)
  
  with open(file_path, "w", encoding="utf-8") as f:
      f.write(raw_code)

In [ ]:
# Gradio UI callback function
def generate_and_zip(query):
    output_dir = "generated_workspace"
    zip_filename = "generated_feature.zip"
    logs = "Starting generation process...\n\n"
    
    # Clean up previous runs
    if os.path.exists(output_dir):
        shutil.rmtree(output_dir)
    os.makedirs(output_dir)
    
    if os.path.exists(zip_filename):
        os.remove(zip_filename)

    # Step 1: Architect Agent makes a plan
    try:
        logs += "1. Generating project plan...\n"
        files_to_create, logs = architect_agent(query, logs)
    except Exception as e:
        return f"Error during planning phase: {str(e)}", None

    # Step 2: Senior Developer Agent executes the Plan and generates code files
    logs += "\n2. Writing code for files...\n"
    
    for file_info in files_to_create:
        filename = file_info['filename']
        description = file_info['description']
        logs += f"   -> Generating {filename}...\n"
        
        try:
            seniordev_agent(filename, description, output_dir)
        except Exception as e:
            logs += f"   -> Error generating {filename}: {str(e)}\n"

    # Step 3: Compress all files into ZIP for Gradio to download
    logs += "\n3. Zipping files...\n"
    try:
        with zipfile.ZipFile(zip_filename, 'w', zipfile.ZIP_DEFLATED) as zipf:
            for root, dirs, files in os.walk(output_dir):
                for file in files:
                    file_path = os.path.join(root, file)
                    # Create the relative path inside the zip file
                    arcname = os.path.relpath(file_path, output_dir)
                    zipf.write(file_path, arcname)
        logs += "\nProcess Complete! You can download your ZIP file below."
    except Exception as e:
        return f"Error zipping files: {str(e)}", None

    return logs, zip_filename



In [ ]:
# Build the Gradio UI
with gr.Blocks(theme=gr.themes.Soft()) as demo:
    gr.Markdown(
        """
        # 🚄 Ruby on Rails Expert Agents
        Enter what you want to build. The AI Agents will formulate a plan, write the files, and give you a ZIP containing the final codebase.
        """
    )
    
    with gr.Row():
        with gr.Column(scale=1):
            query_input = gr.Textbox(
                label="Feature Query", 
                lines=5, 
                placeholder="E.g. Create an online Shop with Products, Categories, Orders and Customers including a REST API."
            )
            generate_btn = gr.Button("Generate & Download Code", variant="primary")
            
        with gr.Column(scale=1):
            log_output = gr.Textbox(label="Execution Logs & Plan", lines=12, interactive=False)
            file_output = gr.File(label="Download Generated Source Code")

    # Wire up the button
    generate_btn.click(
        fn=generate_and_zip,
        inputs=[query_input],
        outputs=[log_output, file_output]
    )

# Launch in the Jupyter Notebook
demo.launch(debug=True, inbrowser=True)
